# 3 — Path clusters

Loads the tables built by **notebook 1** and looks at the **path clustering** — how each trial's
trajectory is grouped into *Direct* vs *Corner-dwelling* (with an occasional *Exploratory* middle).
**It reads no logs and no video.**

The grouping is already done upstream (`task_*/cluster_paths.py`, run as a library inside notebook 1):
each trial is reduced to four PATH-GEOMETRY features — **efficiency, speed variability, time in corner,
trial length** — standardised **per session**, then KMeans-split; the highest-efficiency group is named
*Direct*, the lowest *Corner-dwelling*. Every trial already carries `cluster` / `cluster_name`. This
notebook only **draws** that.

**Both current tasks are shown, one under the other, so they can be compared** — `banish_multiplier`
and `timeout_multiplier` (never pooled; a world fixes the path geometry). **Descriptive only.**

Order: 1 load · 2 counts + defining features · 3 the 2-D feature view · 4 example
trajectories (trial_report style) · 5 per animal, then averaged across animals.

## Load

In [ ]:
MAIN_DIR = '/mnt/server/data'
PIPELINE_DIR = None
# =============================================================================
import sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'results'))

import build_log_df as bl, plot_clusters as pc
bl = importlib.reload(bl); pc = importlib.reload(pc)

_LOCAL = Path('~/repo/session_pipeline_output').expanduser()
OUT = _LOCAL if (_LOCAL / 'df_trials.pkl').exists() else (MAIN_DIR / 'df_log')
print(f'loading dataset from {OUT}')
df_sessions = bl.load(OUT / 'df_sessions.pkl')
df_trials   = bl.load(OUT / 'df_trials.pkl')
print(f'\n{len(df_trials)} trials, {df_sessions.mouse.nunique()} animal(s)')
if 'cluster_name' in df_trials.columns:
    print('cluster_name values:', df_trials.cluster_name.value_counts(dropna=False).to_dict())
else:
    print('!! no cluster_name column -- rerun notebook 1 (the clustering step did not write it)')

## 1 — Everything, per animal

**No manual pick.** Both tasks are drawn from the whole dataset; each animal is kept separate, and the
across-animal *average* uses the animal as the unit (equal weight per animal), so a prolific animal
does not dominate. Set `ANIMALS` only if you want to restrict to a subset.

In [ ]:
ANIMALS = None                  # None = every animal; or a list e.g. ['JPAS_0168', 'JPAS_0231']
TASKS   = ['banish_multiplier', 'timeout_multiplier']   # each drawn as its own row of figures
# =============================================================================
def _sel(task):
    T = df_trials[df_trials.task == task] if 'task' in df_trials.columns else df_trials
    if ANIMALS: T = T[T.mouse.isin(ANIMALS)]
    return pc.clustered(T)          # real path clusters only (drops escape/degenerate/unclustered)

Cs = {t: _sel(t) for t in TASKS}
for t, Ct in Cs.items():
    if len(Ct):
        na = Ct.mouse.nunique() if 'mouse' in Ct.columns else 1
        print(f'{t:20s} {len(Ct):4d} clustered trials, {na} animal(s)   '
              f'{Ct.cluster_name.value_counts().to_dict()}')
    else:
        print(f'{t:20s}   no clustered trials in the selection')

## 2 — The clusters: how many, and what defines them

**One row per task** (banish, timeout): trial count per cluster, then the four **path-geometry features** that produced the split.
*Direct* = high efficiency + low corner-time + short; *Corner-dwelling* = the opposite. These four
features ARE the clustering input, which is why they are the ones shown.

In [ ]:
fig = pc.cluster_overview_by_task(Cs, title='Path clusters by task (banish vs timeout)')
plt.show()

## 3 — The clusters in 2-D (real feature units)

**Not the PCA** — clustering is per-session, so the stored `cluster_pca1/2` live in each session's own
basis and become an unseparable blob once sessions are pooled. The raw features (efficiency, corner
time, trial length) are the same units across sessions, so THEY show the separation.

**One row per task** so banish and timeout can be compared directly.

In [ ]:
fig = pc.cluster_scatter_by_task(Cs, title='Cluster feature space by task (banish vs timeout)')
plt.show()

## 4 — Example trajectories per cluster (trial_report style)

Drawn like the JPAS_0168 `trial_report.pdf` path panel: arena box, plasma path (dark = start →
bright = collection), facing arrows at turns, and the on-screen icon landscape (* reward / X banish /
o unbanish) with the collected icon ringed. Two rows kept: **Direct vs Corner-dwelling**. (Licking
dots are omitted — they are video-derived and not in `df_trials`.)

In [ ]:
for t, C in Cs.items():
    if len(C):
        print(f'==== {t} ====')
        fig = pc.example_paths(C, n=5, title=f'{t} — example paths (Direct vs Corner-dwelling)')
        plt.show()

## 5 — Per animal, then averaged across animals

The headline is the **cluster share per animal** with an **ALL = mean across animals** group (equal
weight per animal) — the "per animal, then average with the animal as the unit" view. Below it, the
share by **outcome** (do banishments / timeouts come from corner-dwelling paths?) and by **world**.

In [ ]:
for t, Ct in Cs.items():
    if not len(Ct): continue
    print(f'==== {t} ====')
    fig = pc.cluster_share_by_animal(Ct, title=f'{t} — cluster share per animal (+ across-animal average)')
    plt.show()
    for by in ['outcome', 'world']:
        if by in Ct.columns and Ct[by].nunique() > 1:
            fig = pc.cluster_composition(Ct, by=by, title=f'{t} — cluster share by {by}')
            plt.show()